# City GNN comparison: GCN / GAT with and without map features vs IDW, kriging and regression-kriging

Self-contained notebook for the confirmatory part of the project. It replaces `run_city_gnn.py`, `d0_city.py` and `rmbench_city.py` (do not import those).
It still needs the project modules in the working folder: `rmbench_core.py`, `rmbench_dmfix.py`, `rmbench_ext.py`, `rmbench_ext2.py`, `rmbench_ext5.py`, `rmbench_ext6.py`,
plus the already built pool (`city_manifest.csv`, `rmbench_cache_city/`, `results_city/prereg_city.json`).

**Order:** 1 setup -> 2 functions -> (3 dev, optional, exploratory) -> 4 freeze -> 5 confirm (ONCE) -> 6 confirm2 (optional replication) -> 7 report-only / per-city tables.

Rules fixed before any confirmatory result: library defaults for every neural arm (no tuning), 150 steps; statistical unit = the city; paired relative-RMSE gains with a
bootstrap CI over cities; **Holm within each family**; densities 30 / 100 / 300 labels. Nothing in the dev run may be used to change an arm.

## 1. Setup

In [1]:
import os, sys, re, json, time, hashlib, glob
import numpy as np, pandas as pd
sys.path.insert(0, os.getcwd())
import rmbench_core as rc
import rmbench_dmfix                      # carrier-frequency fix; keep it before any scenario is loaded
import rmbench_ext as rx, rmbench_ext2 as r2, rmbench_ext5 as x5, rmbench_ext6 as x6

# Must equal the config the units were BUILT with (rmbench_city.CITY_CFG).
CFG = dict(rc.DEFAULT_CFG, cache_dir="rmbench_cache_city", max_points=12000, min_unit_points=1000, n_test=500, densities=(30, 100, 300), n_rep=5)
MANIFEST = "city_manifest.csv"
OUT = "results_city"
PLAN = os.path.join(OUT, "prereg_city_gnn.json")
ROLES = ("source", "val", "dev", "confirm", "confirm2")
os.makedirs(OUT, exist_ok=True)

N_JOBS = min(8, max(1, (os.cpu_count() or 2) // 2))     # memory was ~10 of 15 GB in use earlier: do not go above 8
assert os.path.exists(MANIFEST), f"{MANIFEST} not found in {os.getcwd()}: run this notebook from the project folder"
print("working dir:", os.getcwd(), "| n_jobs:", N_JOBS)

working dir: /home/praaneshnair/gitProjects/prism | n_jobs: 8


## 2. Pool, jobs, contrasts, run, report

In [2]:
def pool(manifest=MANIFEST, cache=CFG["cache_dir"]):
    """{role: [layout = [(unit name, cache dir), ...], ...]} in numeric city order. Roles come from city_manifest.csv (frozen by the city-pool step)."""
    m = pd.read_csv(manifest); role = dict(zip(m.slug, m.role)); by = {}
    for f in sorted(glob.glob(os.path.join(cache, "units", "*.npz"))):
        name = os.path.basename(f)[:-4].replace("__", "|"); slug = name.split("|")[1]
        if role.get(slug) in ROLES: by.setdefault(slug, []).append(name)
    idx = lambda s: (int(re.match(r"city_(\d+)_", s).group(1)) if re.match(r"city_(\d+)_", s) else 10 ** 6, s)
    out = {r: [] for r in ROLES}
    for slug in sorted(by, key=idx): out[role[slug]].append([(n, cache) for n in sorted(by[slug])])
    return out

def units_of(role):
    return [n for L in pool()[role] for n, _ in L]

# ---- pre-declared families (A vs B; gain > 0 => A better), Holm WITHIN each family --------------------------------------------------
FAMILIES = dict(
    primary=[("gcn_knn_side", "rk_side"), ("gat_del_side", "rk_side")],                                  # a GNN with map features beats regression-kriging on the same features
    gnn_feat=[("gat_del_side", "gat_del"), ("gcn_knn_side", "gcn_knn")],                                 # do the map features help the GNN?
    gnn_vs_classical=[("gat_del_side", "idw"), ("gcn_knn_side", "idw"), ("gat_del_side", "krig_global"), ("gcn_knn_side", "krig_global")],
    feat=[("rk_side", "idw"), ("rk_side", "krig_global"), ("rk_gbm", "rk_side")],                        # do the map features help classical interpolation?
    gnn_geo=[("gcn_knn", "idw"), ("gat_del", "idw")],                                                    # geometry-only GNNs vs IDW
    topo=[("gcn_del", "gcn_knn"), ("gat_del", "gat_knn"), ("gat_knn", "gcn_knn"), ("gat_del", "gcn_del")])
POSITIVE = FAMILIES["primary"]     # PRE-DECLARED positive GNN result: either arm beats rk_side, verdict 'A better' (CI > 0), p_holm < 0.05 in this family, at >= 1 density

def min_p_holm(n_clusters, n_tests):
    """Smallest Holm-adjusted p an exact sign-flip test over n_clusters cities can reach in a family of n_tests rows."""
    return min(1.0, 2.0 / 2 ** n_clusters * n_tests)

def make_jobs(gnn=True):
    within = [j for j in x6.within_jobs(gated=False) if j["name"] in ("rk_spline", "rk_gbm")]
    if not gnn: return rx.base_jobs(with_kgnn=False) + within + [j for j in r2.side_jobs() if j["name"] == "rk_side"]
    return rx.base_jobs() + within + r2.side_jobs() + rx.graph_jobs(("gcn_knn", "gcn_del", "gat_knn", "gat_del"))

def run(role, tag, jobs, n_jobs=N_JOBS, densities=None, n_rep=None, out=OUT):
    dens = densities or CFG["densities"]; reps = range(n_rep or CFG["n_rep"]); units = units_of(role)
    if not units: raise RuntimeError(f"no units with role {role!r}: check city_manifest.csv roles and rmbench_cache_city/")
    print(f"[{tag}] role={role}: {len(units)} units in {len(set(u.split('|')[1] for u in units))} cities, densities {tuple(dens)}, reps {len(reps)}, {len(jobs)} jobs: {[j['name'] for j in jobs]}")
    tasks = x5.make_tasks5(units, dens, reps, jobs, CFG, prefix=tag); path = os.path.join(out, f"{tag}.jsonl"); x6.run_tasks6(tasks, path, n_jobs=n_jobs)
    return path

def load_ok(path):
    df = rc.read_results(path); return df[df.status == "ok"]

def per_city_table(path, names=None):
    """Descriptive: RMSE (dB) per city, density and method (units and replicates averaged within a city)."""
    u = x5.unit_rmse(load_ok(path)); t = u.groupby(["parent", "density", "name"]).rmse.mean().unstack("name")
    return t[[n for n in names if n in t.columns]] if names else t

def report(path, strict=True):
    x5.check_run(path, max_fail_frac=0.01, strict=strict); df = load_ok(path)
    u = x5.unit_rmse(df); t = u.groupby(["parent", "density", "name"]).rmse.mean().groupby(["density", "name"]).mean().unstack("density")
    print("\nRMSE (dB), mean over cities (units and replicates averaged within a city):"); print(t.round(3).to_string())
    cts = []
    for fam, cs in FAMILIES.items():
        ct = x5.contrast_table(df.assign(family=fam, split="random", scenario="city"), cs, by=("family", "density"))
        if len(ct): cts.append(ct)
    if not cts: print("no contrasts could be computed"); return None
    ct = pd.concat(cts, ignore_index=True)
    ct["p_holm"] = ct.groupby("family").p.transform(lambda p: pd.Series(rc.holm(p.values), index=p.index))
    ct["p_min_possible"] = [min_p_holm(n, int((ct.family == f).sum())) for n, f in zip(ct.n_clusters, ct.family)]; pd.set_option("display.width", 250)
    print("\npaired contrasts (relative RMSE gain %, A better if > 0; CI = bootstrap over cities; Holm within each family; p_min_possible = smallest p_holm this many cities can give):")
    print(ct[["family", "density", "A", "B", "n_clusters", "mean_gain_pct", "ci_lo", "ci_hi", "mean_gain_db", "p_holm", "p_min_possible", "verdict"]].round(3).to_string(index=False))
    pos = ct[[(a, b) in POSITIVE for a, b in zip(ct.A, ct.B)]]; hit = pos[(pos.verdict == "A better") & (pos.p_holm < 0.05)]
    print(f"\nPRE-DECLARED POSITIVE GNN RESULT (side-feature GNN beats rk_side, CI > 0, p_holm < 0.05 in the primary family at >= 1 density): {'YES' if len(hit) else 'NO'}")
    if len(hit): print(hit[["A", "B", "density", "mean_gain_pct", "ci_lo", "ci_hi", "p_holm"]].round(3).to_string(index=False))
    ct.to_csv(path.replace(".jsonl", "_contrasts.csv"), index=False); return ct

def plan_hash(jobs):
    plan = dict(jobs=jobs, families=FAMILIES, positive=POSITIVE, cfg={k: v for k, v in CFG.items() if k != "cache_dir"}, confirm=sorted(units_of("confirm")))
    return plan, hashlib.sha256(json.dumps(plan, sort_keys=True, default=str).encode()).hexdigest()

P = pool(); print({r: f"{len(L)} cities / {sum(map(len, L))} units" for r, L in P.items()})

{'source': '30 cities / 51 units', 'val': '6 cities / 10 units', 'dev': '7 cities / 12 units', 'confirm': '10 cities / 18 units', 'confirm2': '0 cities / 0 units'}


## 3. DEV run (optional, exploratory only)
Runs the 7 dev cities. Already done once (`results_city/city_gnn_dev.jsonl`); re-running resumes and does nothing if it is complete.
`DEV_DENSITIES` / `DEV_N_REP` exist for quick smoke tests only. **Nothing here is confirmatory and nothing here may change an arm.**

In [3]:
RUN_DEV = False
NO_GNN = False                 # True = classical arms only (fast)
DEV_DENSITIES, DEV_N_REP = None, None
if RUN_DEV:
    tag = "city_gnn_dev" + ("_nognn" if NO_GNN else "")
    p = run("dev", tag, make_jobs(not NO_GNN), N_JOBS, DEV_DENSITIES, DEV_N_REP)
    report(p, strict=False); print("\nDEV run: exploratory only.")
else:
    print("RUN_DEV is False: set it to True to (re)run the dev cities")

RUN_DEV is False: set it to True to (re)run the dev cities


## 4. FREEZE the confirmatory plan
Writes `results_city/prereg_city_gnn.json` (a SHA-256 of the jobs, contrasts, config and the confirm-city list). It **refuses** if any family cannot reach p_holm < 0.05 with the number of confirm cities
(the exact sign-flip test over n cities cannot go below 2/2^n). Re-running is harmless when nothing changed.

In [9]:
FREEZE = True
if FREEZE:
    plan, h = plan_hash(make_jobs(True)); ncl = len(set(n.split("|")[1] for n in plan["confirm"]))
    att = {f: round(min_p_holm(ncl, len(cs) * len(CFG["densities"])), 4) for f, cs in FAMILIES.items()}
    print(f"confirm cities: {ncl}; smallest attainable p_holm per family: {att}")
    bad = {f: p_ for f, p_ in att.items() if p_ >= 0.05}
    assert not bad, f"refusing to freeze: these families cannot reach p_holm < 0.05 with {ncl} confirm cities: {bad}"
    if os.path.exists(PLAN): assert json.load(open(PLAN))["sha256"] == h, "plan differs from the frozen one"
    json.dump(dict(sha256=h, plan=plan, frozen_at=time.strftime("%Y-%m-%d %H:%M:%S")), open(PLAN, "w"), indent=1, default=str); print("frozen", h[:16], "->", PLAN)
else:
    print("FREEZE is False: set it to True once, after reading the table above")

confirm cities: 10; smallest attainable p_holm per family: {'primary': 0.0117, 'gnn_feat': 0.0117, 'gnn_vs_classical': 0.0234, 'feat': 0.0176, 'gnn_geo': 0.0117, 'topo': 0.0234}
frozen 8e48c85156c3d677 -> results_city/prereg_city_gnn.json


## 5. CONFIRM (run ONCE on the 10 confirm cities)
Refuses unless the plan is frozen and unchanged. The run is resumable if interrupted (same task ids); do not change anything between the freeze and this cell.

In [14]:
RUN_CONFIRM = True        # set to True for the single confirmatory run
if RUN_CONFIRM:
    assert os.path.exists(PLAN), "freeze the plan first (section 4)"
    _, h = plan_hash(make_jobs(True)); assert json.load(open(PLAN))["sha256"] == h, "the plan or the confirm cities changed after freezing"
    p_confirm = run("confirm", "city_gnn_confirm", make_jobs(True), N_JOBS)
    ct_confirm = report(p_confirm, strict=True)
else:
    print("RUN_CONFIRM is False")

[city_gnn_confirm] role=confirm: 18 units in 10 cities, densities (30, 100, 300), reps 5, 13 jobs: ['trend', 'idw', 'krig_global', 'kgnn', 'rk_spline', 'rk_gbm', 'gcn_knn_side', 'gat_del_side', 'rk_side', 'gcn_knn', 'gcn_del', 'gat_knn', 'gat_del']
[run_tasks6] 270 tasks, 0 done, 270 to run (n_jobs=8) -> results_city/city_gnn_confirm.jsonl
  25/270 101s failed jobs: 0
  50/270 189s failed jobs: 0
  75/270 294s failed jobs: 0
  100/270 457s failed jobs: 0
  125/270 644s failed jobs: 0
  150/270 816s failed jobs: 0
  175/270 966s failed jobs: 0
  200/270 1077s failed jobs: 0
  225/270 1208s failed jobs: 0
  250/270 1403s failed jobs: 0
  270/270 1528s failed jobs: 0
[check_run] results_city/city_gnn_confirm.jsonl: 3510 rows, 0 failed (0.00%)

RMSE (dB), mean over cities (units and replicates averaged within a city):
density         30     100    300
name                             
gat_del       6.113  4.867  3.438
gat_del_side  4.572  3.414  2.846
gat_knn       6.190  4.789  3.417
gcn_

## 6. CONFIRM2 (optional replication)
Cities that became eligible after the freeze (none yet: the downloads were rate-limited). Same frozen jobs and contrasts; run only after section 5.

In [15]:
RUN_CONFIRM2 = False
if RUN_CONFIRM2:
    assert os.path.exists(PLAN), "freeze the plan first (section 4)"
    _, h = plan_hash(make_jobs(True)); assert json.load(open(PLAN))["sha256"] == h, "the plan or the confirm cities changed after freezing"
    p_confirm2 = run("confirm2", "city_gnn_confirm2", make_jobs(True), N_JOBS)
    ct_confirm2 = report(p_confirm2, strict=True)
else:
    print("RUN_CONFIRM2 is False")

RUN_CONFIRM2 is False


## 7. Report-only and per-city tables (no computation of new fits)

In [16]:
REPORT_FILE = os.path.join(OUT, "city_gnn_dev.jsonl")      # or city_gnn_confirm.jsonl / city_gnn_confirm2.jsonl
if os.path.exists(REPORT_FILE):
    _ = report(REPORT_FILE, strict=False)
else:
    print("not found:", REPORT_FILE)

[check_run] results_city/city_gnn_dev.jsonl: 2340 rows, 0 failed (0.00%)

RMSE (dB), mean over cities (units and replicates averaged within a city):
density         30     100    300
name                             
gat_del       5.878  4.097  3.179
gat_del_side  4.366  3.302  2.637
gat_knn       6.065  4.024  3.089
gcn_del       5.976  4.233  3.213
gcn_knn       6.057  4.119  3.138
gcn_knn_side  4.486  3.362  2.780
idw           5.311  3.739  2.831
kgnn          5.086  3.542  2.603
krig_global   5.252  3.614  2.670
rk_gbm        4.125  2.923  2.372
rk_side       4.009  3.157  2.523
rk_spline     4.160  3.121  2.483
trend         7.034  6.457  6.429

paired contrasts (relative RMSE gain %, A better if > 0; CI = bootstrap over cities; Holm within each family; p_min_possible = smallest p_holm this many cities can give):
          family  density            A           B  n_clusters  mean_gain_pct   ci_lo  ci_hi  mean_gain_db  p_holm  p_min_possible      verdict
         primary       30

In [17]:
TABLE_FILE = os.path.join(OUT, "city_gnn_dev.jsonl")
if os.path.exists(TABLE_FILE):
    pd.set_option("display.width", 250)
    print(per_city_table(TABLE_FILE, ["idw", "krig_global", "rk_side", "rk_gbm", "gat_del_side", "gcn_knn_side", "gat_del", "gcn_knn"]).round(2).to_string())
else:
    print("not found:", TABLE_FILE)

name                                  idw  krig_global  rk_side  rk_gbm  gat_del_side  gcn_knn_side  gat_del  gcn_knn
parent                      density                                                                                  
city_13_columbus_3p5        30       6.58         6.61     5.56    5.75          5.59          5.59     7.19     7.45
                            100      4.40         4.17     4.26    3.59          3.73          3.83     4.92     5.23
                            300      3.24         2.92     3.20    2.70          3.01          3.11     3.65     3.65
city_26_dubai_3p5           30       3.88         3.85     2.78    3.29          3.08          3.38     3.97     3.89
                            100      3.24         3.09     2.44    2.54          3.24          2.59     3.47     3.38
                            300      2.24         2.21     1.78    1.85          2.39          2.24     2.86     2.62
city_53_sankt-peterburg_3p5 30       2.61         2.42  